# Chapter 5: Synchronous Communication via MCP

The **Model Context Protocol (MCP)** is Anthropic's open standard for connecting
LLMs to external tools, data sources, and reusable prompts via a structured JSON-RPC interface.

**Concepts:**
- Tools: stateful actions that agents can invoke
- Resources: data sources exposed via URI
- Prompts: reusable, parameterized prompt templates
- Server lifecycle and transport (stdio / HTTP+SSE)

In [ ]:
import sys, os, asyncio, json
sys.path.insert(0, os.path.abspath('../..'))
from core.mcp.server import MCPServer, ToolDefinition, ResourceDefinition, PromptDefinition
print('MCP server module ready.')

## 5.1 Registering and Calling Tools

A **Tool** represents a callable action with a declared JSON Schema input.
The LLM uses the schema to populate arguments; MCP validates them before dispatch.
> MCP is *synchronous from the LLM's perspective*: the model pauses until the tool returns.

In [ ]:
import asyncio

# Simulate a real order database
ORDERS = {
    'ORD-001': {'id': 'ORD-001', 'status': 'shipped', 'customer': 'alice@example.com', 'total': 89.99},
    'ORD-002': {'id': 'ORD-002', 'status': 'processing', 'customer': 'bob@example.com', 'total': 142.50},
}

async def get_order_handler(order_id: str) -> dict:
    if order_id not in ORDERS:
        raise ValueError(f'Order {order_id} not found')
    return ORDERS[order_id]

async def list_orders_handler(limit: int = 10) -> list:
    return list(ORDERS.values())[:limit]

# Create MCP server
server = MCPServer('OrderMCPServer', version='1.0.0')

server.register_tool(ToolDefinition(
    name='get_order',
    description='Retrieve an order by its ID',
    input_schema={
        'type': 'object',
        'properties': {'order_id': {'type': 'string', 'description': 'The order identifier'}},
        'required': ['order_id'],
    },
    handler=get_order_handler,
))

server.register_tool(ToolDefinition(
    name='list_orders',
    description='List all orders with an optional limit',
    input_schema={
        'type': 'object',
        'properties': {'limit': {'type': 'integer', 'description': 'Max results', 'default': 10}},
    },
    handler=list_orders_handler,
))

async def demo_tools():
    # List available tools (what the LLM sees)
    print('Available tools:')
    for tool in server.list_tools():
        print(f'  - {tool["name"]}: {tool["description"]}')
    print()

    # Call a tool
    result = await server.call_tool('get_order', {'order_id': 'ORD-001'})
    print('Tool call result (get_order ORD-001):')
    print(json.dumps(result, indent=2, default=str))
    print()

    # Error handling
    err = await server.call_tool('get_order', {'order_id': 'ORD-MISSING'})
    print('Error response (missing order):')
    print(json.dumps(err, indent=2))

asyncio.run(demo_tools())

## 5.2 Exposing Resources via URI

**Resources** let agents read structured data using addressable URIs.
Think of them as `GET` endpoints that return typed content (JSON, text, markdown).

In [ ]:
import asyncio, json

# Register data resources
server.register_resource(ResourceDefinition(
    uri='orders://catalog',
    name='Order Catalog',
    mime_type='application/json',
    description='Full product catalog with current prices',
    reader=lambda: inventory_reader(),
))

async def inventory_reader():
    return json.dumps([
        {'sku': 'SKU-001', 'name': 'Laptop Stand', 'price': 49.99, 'stock': 150},
        {'sku': 'SKU-002', 'name': 'Mechanical Keyboard', 'price': 129.99, 'stock': 42},
        {'sku': 'SKU-003', 'name': 'USB-C Hub', 'price': 39.99, 'stock': 200},
    ])

server.register_resource(ResourceDefinition(
    uri='orders://catalog',
    name='Order Catalog',
    mime_type='application/json',
    description='Full product catalog',
    reader=inventory_reader,
))

async def demo_resources():
    print('Available resources:')
    for r in server.list_resources():
        print(f'  - {r["uri"]}: {r["name"]} ({r["mimeType"]})')
    print()

    content = await server.read_resource('orders://catalog')
    catalog = json.loads(content['contents'][0]['text'])
    print('Catalog contents:')
    for item in catalog:
        print(f'  {item["sku"]}: {item["name"]} £{item["price"]} (stock: {item["stock"]})')

asyncio.run(demo_resources())

## 5.3 Reusable Prompt Templates

**Prompts** are version-controlled, parameterized templates stored on the server.
This decouples prompt engineering from application code — update the prompt
without redeploying the agent.

In [ ]:
import asyncio, json

server.register_prompt(PromptDefinition(
    name='order_summary',
    description='Summarize an order for a customer-facing email',
    arguments=[
        {'name': 'order_id', 'description': 'Order identifier', 'required': 'true'},
        {'name': 'customer_name', 'description': 'Customer display name', 'required': 'true'},
    ],
    template="""You are a helpful customer service assistant.
Summarize order {order_id} for customer {customer_name}. 
Be concise, friendly, and include the key details (status, items, total).""",
))

async def demo_prompts():
    print('Available prompts:')
    for p in server.list_prompts():
        print(f'  - {p["name"]}: {p["description"]}')
    print()

    # Materialize a prompt with concrete values
    prompt = await server.get_prompt(
        'order_summary',
        {'order_id': 'ORD-001', 'customer_name': 'Alice'}
    )
    print('Materialized prompt:')
    print(prompt['messages'][0]['content']['text'])

asyncio.run(demo_prompts())

## 5.4 Production Deployment on Azure

Production MCP deployment on Azure uses the official `mcp` Python SDK with HTTP+SSE transport,
hosted inside an **Azure Container Apps** revision with HTTPS provided by the managed environment:

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP('OrderMCPServer')

@mcp.tool()
async def get_order(order_id: str) -> dict:
    # Azure Cosmos DB lookup
    return await cosmos.get_item(order_id)

if __name__ == '__main__':
    mcp.run(transport='streamable-http', host='0.0.0.0', port=8080)
```

Configure Azure OpenAI to use your MCP server via the `tool_choice` API (≥ 2025-04-01 preview).